In [ ]:
import os

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader

import models_mae

# parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

imagenet_mean = np.array([0.485, 0.456, 0.406])
imagenet_std = np.array([0.229, 0.224, 0.225])


# channel encoder and decoder
class autoencoder(torch.nn.Module):

    def __init__(self):
        super(autoencoder, self).__init__()
        self.fc1 = torch.nn.Linear(1024, 128)
        self.fc2 = torch.nn.Linear(128, 1024)

    def forward(self, x):
        if x.shape[-1] == 1024:
            return self.fc1(x)
        else:
            return self.fc2(x)


# semantic model
def prepare_model(chkpt_dir, ch_path="", arch="mae_vit_large_patch16"):
    model = getattr(models_mae, arch)()
    checkpoint = torch.load(chkpt_dir, map_location="cpu")
    msg = model.load_state_dict(checkpoint["model"], strict=False)
    print(msg)
    model = model.to(device)
    ae = autoencoder().to(device)
    if ch_path != "":
        ae.load_state_dict(torch.load(ch_path))
        print("channel loaded")
    return model, ae


# noise channel
def AWGN_channel(x, snr):
    [batch_size, length, dim] = x.shape
    x_power = torch.sum(torch.abs(x)) / (batch_size * length * dim)
    n_power = x_power / (10**(snr / 10.0))
    noise = torch.rand(batch_size, length, dim, device="cuda") * n_power
    return x + noise


# dataclass
class ImageDataset(torch.utils.data.Dataset):

    def __init__(self, path):
        self.imgs = os.listdir(path)
        self.path = path

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_name = os.path.join(self.path, self.imgs[idx])
        img = Image.open(img_name).convert("RGB")
        img = img.resize((224, 224))
        img = np.array(img) / 255.0
        img = img - imagenet_mean
        img = img / imagenet_std

        return torch.tensor(img).permute(2, 0, 1).float()


def getloader(path, batch_size):
    dataset = ImageDataset(path)
    dataloader = DataLoader(dataset,
                            batch_size=batch_size,
                            shuffle=True,
                            drop_last=True,
                            num_workers=0)
    return dataloader

In [ ]:
loader = getloader("data/train/train_224", 16)
next(iter(loader)).shape

In [ ]:
model, ae = prepare_model(
    chkpt_dir="./ckpt/mae_visualize_vit_large_ganloss.pth")
optimizer = optim.Adam(ae.parameters(), lr=2e-4)
loss_func = nn.MSELoss()

snr_list = [0, 5, 10, 15, 20, 25]
for snr in snr_list:
    for epoch in range(50):
        total_loss = 0
        for i, x in enumerate(loader):
            x = x.to(device)
            latent, _, _ = model.forward_encoder(x, mask_ratio=0.75)
            channel_enc = ae(latent)
            noise_latent = AWGN_channel(channel_enc, snr=snr)
            channel_dec = ae(noise_latent)
            loss = loss_func(channel_dec, latent)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        torch.save(ae.state_dict(),
                   f"./ckpt/channel/snr_{snr}_epoch_{epoch}.pth")
        print(f"Epoch {epoch} loss: {total_loss / (i + 1)}")
        # 保存loss到对应的文件夹
        with open(f"./ckpt/channel/snr_{snr}_loss.txt", "a") as f:
            f.write(f"Epoch {epoch} loss: {total_loss / (i + 1)}\n")